# init_recording.ipynb — Synchronous Sensor Recording

This notebook **records**. It produces raw data only.

**Output:** `SESSION_DIR/events.jsonl` + `SESSION_DIR/video.mp4`

These files are then processed by `ground_truth_classifier.ipynb`.

**Hardware sources:**
- **Polar H10** (Bluetooth LE) — R-R intervals
- **Camera** (built-in / Continuity) — video for FER
- **ESP32 + AD8232 + SGP40 + SCD4x** (WebSocket on port 81) — plant voltage @ 100Hz, VOC index and CO2 (ppm) per batch

**Mock mode:** Polar and camera always use real hardware. When `USE_MOCK_ESP32 = True`,
plant voltage, VOC and CO2 are synthesised directly into the event store (useful when the
ESP32 is not yet on the network).

**Baseline phase:** the first `BASELINE_S` seconds are a resting phase — all sensors
record, but the camera does not write video/frames yet. `phase` events
(`baseline_start` / `baseline_end`) mark the boundary in `events.jsonl`.
</cell id="cell-0">


## 1  Imports & Configuration

In [ ]:
import os
os.environ["OPENCV_AVFOUNDATION_SKIP_AUTH"] = "1"  # macOS camera fix
import asyncio, json, math, struct, threading, time, warnings
from pathlib import Path

import numpy as np

try:
    from bleak import BleakScanner, BleakClient
    BLEAK_OK = True
except ImportError:
    BLEAK_OK = False
    print("bleak not installed — pip install bleak")

try:
    import cv2
    CV2_OK = True
except ImportError:
    CV2_OK = False
    print("opencv not installed — pip install opencv-python")

try:
    import websockets
    WEBSOCKETS_OK = True
except ImportError:
    WEBSOCKETS_OK = False
    print("websockets not installed — pip install websockets")

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION — adjust here
# ══════════════════════════════════════════════════════════════════
import datetime
PROBAND_ID    = "proband_01"            # ← change per session
_ts           = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
SESSION_ID    = f"{_ts}_{PROBAND_ID}"
SESSION_DIR   = Path("./sessions") / SESSION_ID
DURATION_S    = 660                      # recording length in seconds
BASELINE_S    = 60                       # resting phase at the start (no video)
VIDEO_FPS     = 30                       # camera frame rate

# ESP32 plant + VOC streamer (see plant_sensor_script.ino)
ESP32_IP        = "192.168.0.100"        # ← IP shown on the OLED screen
ESP32_PORT      = 81
ESP32_RATE_HZ   = 100                    # output rate the ESP32 sends at

# Mock switch — when True, plant_voltage + voc are synthesised locally
# (Polar and camera still need real hardware)
USE_MOCK_ESP32  = True

SESSION_DIR.mkdir(parents=True, exist_ok=True)
EVENT_STORE = SESSION_DIR / "events.jsonl"
VIDEO_FILE  = SESSION_DIR / "video.mp4"

print(f"Session folder : {SESSION_DIR.resolve()}")
print(f"Duration       : {DURATION_S}s  |  Baseline: {BASELINE_S}s")
print(f"ESP32          : {'MOCK' if USE_MOCK_ESP32 else f'ws://{ESP32_IP}:{ESP32_PORT}'}")


## 2  Event Store

All sensors write independently into one shared JSONL file. Each event has
exactly three fields: `ts` (Unix ms), `sensor`, `value`.

```
{"ts": 1700000000000, "sensor": "phase",        "value": "baseline_start"}
{"ts": 1700000000123, "sensor": "hr_rr",        "value": 812.3}
{"ts": 1700000000200, "sensor": "camera_frame",  "value": 42}
{"ts": 1700000001000, "sensor": "co2",           "value": 412.0}
{"ts": 1700000001000, "sensor": "voc",           "value": 95.0}
{"ts": 1700000001100, "sensor": "plant_voltage", "value": 0.823}
{"ts": 1700000060000, "sensor": "phase",        "value": "baseline_end"}
```

No sensor is the master clock. Resampling onto a common grid happens later in
`ground_truth_classifier.ipynb`.


In [2]:
def write_event(ts_ms, sensor, value):
    """Write one event to the store (thread-safe via append)."""
    with open(EVENT_STORE, "a") as f:
        f.write(json.dumps({"ts": int(ts_ms), "sensor": sensor,
                            "value": value}) + "\n")

# Clear the store (new session)
EVENT_STORE.write_text("")
print(f"Event store initialised: {EVENT_STORE}")


Event store initialised: sessions/2026-06-01_17-04_proband_01/events.jsonl


## 3  Sensor Functions

### 3a  Polar H10 — BLE R-R Intervals

In [ ]:
HR_CHAR = "00002a37-0000-1000-8000-00805f9b34fb"

def polar_handler(sender, data):
    """BLE callback: decode R-R intervals (Bluetooth SIG 2011).
    R-R is encoded as little-endian uint16 in units of 1/1024 s."""
    flags      = data[0]
    hr_16_bit  = flags & 0x01
    rr_present = flags & 0x10
    offset     = 3 if hr_16_bit else 2
    if flags & 0x08:
        offset += 2                        # skip Energy Expended field
    if rr_present:
        while offset + 1 < len(data):
            rr_raw = struct.unpack_from("<H", data, offset)[0]
            rr_ms  = rr_raw / 1024 * 1000  # 1/1024 s → ms
            write_event(time.time() * 1000, "hr_rr", round(rr_ms, 1))
            offset += 2

async def run_polar(duration_s):
    """Connect to the Polar H10 and stream R-R intervals until duration_s
    elapses. Reconnects automatically (scan + connect) if no device is
    found yet or if the BLE link drops mid-session."""
    if not BLEAK_OK:
        raise RuntimeError("bleak not installed")
    t_end = time.time() + duration_s
    while time.time() < t_end:
        print("Scanning for Polar H10 ...")
        devices = await BleakScanner.discover()
        polar   = next((d for d in devices if d.name and "Polar" in d.name), None)
        if polar is None:
            print("No Polar H10 found — retrying in 2s ...")
            await asyncio.sleep(2)
            continue
        print(f"Connected: {polar.name}")
        try:
            async with BleakClient(polar.address) as client:
                await client.start_notify(HR_CHAR, polar_handler)
                remaining = t_end - time.time()
                if remaining > 0:
                    await asyncio.sleep(remaining)
                await client.stop_notify(HR_CHAR)
        except Exception as e:
            print(f"Polar disconnected ({e}); reconnecting ...")
            await asyncio.sleep(2)
    print("Polar done.")


### 3b  Camera — Frame Timestamps from System Clock

In [ ]:
def run_camera(t0_s, duration_s, baseline_s=0):
    import time

    # Prefer index 1 (Mac camera), fall back to 0 (e.g. iPhone Continuity).
    # If only one camera exists, it becomes index 0 → fallback still works.
    selected_index = None
    for i in [1, 0]:
        cap_test = cv2.VideoCapture(i)
        if cap_test.isOpened():
            ok, frame = cap_test.read()
            cap_test.release()
            if ok and frame is not None and frame.size > 0:
                selected_index = i
                print(f"Using camera index {i}")
                break
            else:
                print(f"Camera {i} opened but returned no frames — skipping")

    if selected_index is None:
        print("No working camera found"); return

    cap = cv2.VideoCapture(selected_index)
    time.sleep(2)                       # macOS warm-up
    for _ in range(10):
        cap.read()                      # discard warm-up frames

    # Read resolution from the camera
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 120:
        fps = VIDEO_FPS
    print(f"Resolution: {w}x{h} @ {fps:.0f}fps")

    # avc1 (H.264) works reliably on macOS; fall back to mp4v
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    writer = cv2.VideoWriter(str(VIDEO_FILE), fourcc, fps, (w, h))
    if not writer.isOpened():
        print("avc1 failed — trying mp4v")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(str(VIDEO_FILE), fourcc, fps, (w, h))

    if baseline_s > 0:
        print(f"Baseline: {baseline_s}s rest — camera warm, not recording yet")

    # t0_s anchors baseline/end to the GLOBAL session start (same clock the
    # "phase" events use), not to when camera setup happens to finish —
    # camera selection + warm-up above can itself take several seconds, so
    # using a thread-local start time here would let the actual recording
    # start drift away from the baseline_end boundary in the event stream.
    frame_id = 0
    t_record_start = t0_s + baseline_s
    t_end          = t0_s + duration_s
    while time.time() < t_end:
        ok, frame = cap.read()
        if not ok or frame is None or frame.size == 0:
            time.sleep(0.01); continue
        if time.time() < t_record_start:
            continue                     # baseline: stay warm, don't record
        ts_ms = time.time() * 1000       # per-frame system timestamp
        write_event(ts_ms, "camera_frame", frame_id)
        writer.write(frame)
        frame_id += 1

    cap.release()
    writer.release()
    print(f"Camera done: {frame_id} frames -> {VIDEO_FILE}")


### 3c  Environment Sensors — Plant voltage + VOC + CO2 (ESP32 / WebSocket)

In [ ]:
# ── ESP32 WebSocket listener (plant voltage @ 100Hz + VOC + CO2) ────
#
# Wire protocol: every ~400ms the ESP32 broadcasts a JSON batch:
#     {"type":"data", "timestamp":<ms>, "sampleRate":100,
#      "samples":50, "voc":<index>, "co2":<ppm>, "voltages":[mV, mV, ...]}
#
# The "sampleRate" field is the firmware's *nominal* rate, not what it
# actually achieves (plant_sensor_script.ino's DECIMATION_FACTOR is an
# integer division that truncates, so the real output rate differs from
# the constant it reports). Trusting it would distort per-sample
# timestamps. Instead we derive the real sample spacing from the host
# clock: the actual elapsed time between two batch arrivals divided by
# the number of samples in the batch. The reported sampleRate is only
# used as a fallback for the very first batch.
#
# Per-sample timestamping: the batch arrives at host time T with N
# voltage samples. We back-calculate each sample timestamp as
# T - (N-1-i) * step_ms so that the last sample is stamped at T.
# Voltages are converted from mV to V to match the existing pipeline.

async def esp32_listener(duration_s):
    """Connect to ESP32 over WebSocket, write voltage + VOC + CO2 events
    until duration_s elapses or the recording session ends."""
    if not WEBSOCKETS_OK:
        raise RuntimeError("pip install websockets")
    import websockets
    uri = f"ws://{ESP32_IP}:{ESP32_PORT}"
    t_end = time.time() + duration_s
    n_batches = 0
    prev_arrival_ms = None
    while time.time() < t_end:
        try:
            async with websockets.connect(uri, ping_interval=20, ping_timeout=10) as ws:
                print(f"ESP32 connected: {uri}")
                async for message in ws:
                    if time.time() >= t_end:
                        break
                    try:
                        data = json.loads(message)
                    except Exception:
                        continue
                    if data.get("type") != "data":
                        continue
                    voltages = data.get("voltages", [])
                    n = len(voltages)
                    if n == 0:
                        continue
                    arrival_ms = time.time() * 1000

                    # Real sample spacing from observed batch cadence
                    # (falls back to the reported rate for the first batch).
                    if prev_arrival_ms is not None and arrival_ms > prev_arrival_ms:
                        step_ms = (arrival_ms - prev_arrival_ms) / n
                    else:
                        step_ms = 1000.0 / data.get("sampleRate", ESP32_RATE_HZ)
                    prev_arrival_ms = arrival_ms

                    # Back-calculate per-sample timestamps
                    for i, mv in enumerate(voltages):
                        ts_ms = arrival_ms - (n - 1 - i) * step_ms
                        volts = float(mv) / 1000.0       # mV -> V
                        write_event(ts_ms, "plant_voltage", round(volts, 6))

                    # VOC + CO2 stamped at batch arrival
                    voc = data.get("voc")
                    if voc is not None:
                        write_event(arrival_ms, "voc", round(float(voc), 1))
                    co2 = data.get("co2")
                    if co2 is not None:
                        write_event(arrival_ms, "co2", round(float(co2), 1))

                    n_batches += 1
        except Exception as e:
            print(f"ESP32 disconnected ({e}); retrying in 2s ...")
            await asyncio.sleep(2)
    print(f"ESP32 listener done — {n_batches} batches received")

# ── Mock: synthesise plant + VOC + CO2 directly (no fake WebSocket) ──
_rng_env = np.random.default_rng(99)

def run_mock_esp32(duration_s):
    """Write synthetic plant_voltage @ 100Hz + voc/co2 @ ~2Hz into the
    event store. Used when USE_MOCK_ESP32 = True (e.g. ESP32 not on
    network)."""
    t_end = time.time() + duration_s
    next_env = time.time()
    last_v   = 0.50
    while time.time() < t_end:
        ts_ms = time.time() * 1000
        # plant voltage: slow drift + small noise around 0.5V
        last_v += float(_rng_env.normal(0, 0.001))
        last_v  = float(np.clip(last_v, 0.30, 0.70))
        write_event(ts_ms, "plant_voltage", round(last_v, 6))
        # VOC + CO2 roughly every 500ms (matches real ESP32 batch cadence)
        if time.time() >= next_env:
            voc = 95 + float(_rng_env.normal(0, 6))
            co2 = 450 + float(_rng_env.normal(0, 15))
            write_event(ts_ms, "voc", round(voc, 1))
            write_event(ts_ms, "co2", round(co2, 1))
            next_env = time.time() + 0.5
        time.sleep(0.01)        # 100Hz

def run_env_sensors(duration_s):
    """Dispatcher: real ESP32 listener (plant_voltage + voc + co2) or mock."""
    if USE_MOCK_ESP32:
        run_mock_esp32(duration_s)
    else:
        # Real ESP32 — async WebSocket in its own event loop
        loop = asyncio.new_event_loop()
        try:
            loop.run_until_complete(esp32_listener(duration_s))
        finally:
            loop.close()


## 4  Start Recording

All sensors start at the same time. The system clock (`time.time()`) is the
common time base — no sensor is the master.

**Flow:**
1. Event store already initialised (Block 2)
2. Camera + environment sensors start in their own threads
3. Polar H10 runs in the asyncio loop (BLE requires async)
4. After `DURATION_S` seconds everything stops automatically
5. Short status report

Afterwards: open `events.jsonl` and `video.mp4` in `SESSION_DIR` →
run `ground_truth_classifier.ipynb`.


In [ ]:
if USE_MOCK_ESP32:
    print("=" * 62)
    print("  WARNING: USE_MOCK_ESP32 = True")
    print("  plant_voltage / voc / co2 will be SYNTHETIC, not real")
    print("  sensor data. Set USE_MOCK_ESP32 = False in cell 2 for")
    print("  a real recording session.")
    print("=" * 62)
    print()

if not BLEAK_OK:
    raise RuntimeError("pip install bleak  (Polar H10 requires bleak)")
if not CV2_OK:
    raise RuntimeError("pip install opencv-python  (camera requires opencv)")
if not USE_MOCK_ESP32 and not WEBSOCKETS_OK:
    raise RuntimeError("pip install websockets  (real ESP32 needs the websockets client)")

T0_S  = time.time()
T0_MS = int(T0_S * 1000)
# Save session metadata (important for later synchronisation)
meta = {"session_id": SESSION_ID, "proband_id": PROBAND_ID,
        "t0_ms": T0_MS, "duration_s": DURATION_S, "baseline_s": BASELINE_S,
        "esp32_uri": f"ws://{ESP32_IP}:{ESP32_PORT}" if not USE_MOCK_ESP32 else "mock",
        "recorded_at": datetime.datetime.now().isoformat()}
(SESSION_DIR / "session_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Session start : {T0_MS} ms  ({time.strftime('%H:%M:%S')})")
print(f"Baseline      : first {BASELINE_S}s rest — please sit still, no video")
print(f"Total length  : {DURATION_S}s")
print(f"ESP32         : {'MOCK' if USE_MOCK_ESP32 else f'ws://{ESP32_IP}:{ESP32_PORT}'}")
print()

# Mark the baseline boundary directly in the event stream so
# ground_truth_classifier.ipynb doesn't have to re-derive it from
# session_meta.json.
write_event(T0_MS, "phase", "baseline_start")
if BASELINE_S > 0:
    threading.Timer(
        BASELINE_S,
        lambda: write_event(time.time() * 1000, "phase", "baseline_end")
    ).start()

errors = []

def _run_camera():
    try: run_camera(T0_S, DURATION_S, BASELINE_S)
    except Exception as e: errors.append(f"Camera: {e}")

def _run_env():
    try: run_env_sensors(DURATION_S)
    except Exception as e: errors.append(f"Env (ESP32): {e}")

# Start threads
t_cam = threading.Thread(target=_run_camera, daemon=True)
t_env = threading.Thread(target=_run_env,    daemon=True)
t_cam.start()
t_env.start()

# Polar in the asyncio loop (Jupyter: await usable directly)
await run_polar(DURATION_S)

# Wait for threads to finish
t_cam.join()
t_env.join()

if errors:
    print("\nErrors during recording:")
    for e in errors: print(f"  {e}")
else:
    print("\nRecording complete — no errors")

# Status report
from collections import Counter
events = [json.loads(l) for l in EVENT_STORE.read_text().strip().split("\n") if l.strip()]
by_sensor = Counter(e["sensor"] for e in events)
print(f"\nEvent store: {len(events)} events in {EVENT_STORE.name}")
for s, n in sorted(by_sensor.items()):
    print(f"  {s:<20} {n:>6}")
print(f"\nNext step: run ground_truth_classifier.ipynb")
print(f"SESSION_DIR = '{SESSION_DIR.resolve()}'")
